# GRIFFIN HR Classification Project
**BUAD 5722 / BUAD 5742 — Spring 2026 — Team 7**
**Steven Alvarado, Anmol Motwani, JR Jones, Brynn Vetrano**

# GRIFFIN Environment Setup

> **W&M HR Position Classification & Pay Matching Tool**  
> Created: March 30, 2026 | Run this notebook once to set up the project environment.

This notebook creates a **conda environment** (`griffin`) and installs all
dependencies needed for the GRIFFIN project pipeline:

| Layer | Tools | Purpose |
|-------|-------|---------|
| Scraping & Parsing | requests, BeautifulSoup, pdfplumber, lxml | Workday API + DHRM pages |
| Data & Database | pandas, sqlalchemy, pymysql, openpyxl | DataFrames, SQLite/MySQL, Excel I/O |
| Visualization | streamlit, plotly, seaborn, matplotlib | App UI + notebook charts |
| ML Explainability | shap | SHAP values for H2O model interpretability |
| LLM / Agentic AI | langchain, langchain-core, langgraph | Multi-agent classification (Assignment 5 + Streamlit) |

### Why conda instead of venv?

1. **Non-Python dependencies.** When we add H2O later, conda can install Java (`openjdk`)
   directly into the environment -- no manual JDK download or `JAVA_HOME` setup.
2. **Binary compatibility.** Packages like `shap`, `numpy`, and `scipy` ship compiled
   C/Fortran code. Conda's solver ensures all compiled pieces are built against the same
   libraries, preventing import-time crashes from mismatched binaries.
3. **Consistency with other courses.** You already use conda for BUAD5022, CTBA, etc.
   Same workflow, same `conda activate`, same muscle memory.

**Not installed here:** `h2o` -- follow the **Big Data course lab setup process** for H2O installation. The lab walkthrough covers Java dependencies and troubleshooting.

---
## 1. Locate Conda

Your active conda is the **Anaconda** install at `C:\ProgramData\anaconda3\`. This is the
one your terminal uses when you type `conda activate`. Environments created by this conda
get proper names in `conda env list` (like `big_data_mid`, `BUAD5022`, etc.).

We reference the full path to `conda.exe` so this notebook works regardless of which
kernel runs it.

In [1]:
import subprocess, sys, os, json

# Full path to the ANACONDA conda (your active install, manages named envs)
# NOT the Miniconda one at C:\Users\salva\miniconda3\ -- that creates nameless envs
CONDA = r"C:\ProgramData\anaconda3\Scripts\conda.exe"
ENV_NAME = "griffin"

# Verify conda is reachable
result = subprocess.run([CONDA, "--version"], capture_output=True, text=True)
print(result.stdout.strip())
print(f"Conda path: {CONDA}")
print(f"Target env: {ENV_NAME}")

conda 25.5.1
Conda path: C:\ProgramData\anaconda3\Scripts\conda.exe
Target env: griffin


---
## 2. Create the Conda Environment

A **conda environment** is an isolated directory containing its own Python interpreter,
packages, and (optionally) non-Python tools like Java. Environments created by Anaconda
live at `C:\ProgramData\anaconda3\envs\` alongside your existing ones (`big_data_mid`,
`nyctaxi_env`, etc.). They show up with proper names in `conda env list` and are visible
to VS Code, terminal Jupyter, and Anaconda Navigator automatically.

We use Python 3.11 because it has the broadest compatibility with our dependency stack.
H2O, SHAP, and some LangChain extensions don't yet fully support 3.13.

In [2]:
# Check if the environment already exists
result = subprocess.run(
    [CONDA, "env", "list", "--json"],
    capture_output=True, text=True,
)
env_list = json.loads(result.stdout)
env_names = [os.path.basename(p) for p in env_list["envs"]]

if ENV_NAME in env_names:
    print(f"Environment '{ENV_NAME}' already exists -- skipping creation.")
    print("To start fresh: conda remove -n griffin --all")
else:
    print(f"Creating conda environment '{ENV_NAME}' with Python 3.11 ...")
    subprocess.run(
        [CONDA, "create", "-n", ENV_NAME, "python=3.11", "-y"],
        check=True,
    )
    print("Done.")

Environment 'griffin' already exists -- skipping creation.
To start fresh: conda remove -n griffin --all


In [2]:
# Get the path to the environment's Python and pip
# On Windows: ~/.conda/envs/griffin/python.exe  (or miniconda3/envs/griffin/python.exe)
result = subprocess.run(
    [CONDA, "run", "-n", ENV_NAME, "python", "-c",
     "import sys; print(sys.executable)"],
    capture_output=True, text=True, check=True,
)
ENV_PYTHON = result.stdout.strip()
ENV_DIR = os.path.dirname(ENV_PYTHON)
ENV_PIP = os.path.join(ENV_DIR, "Scripts", "pip.exe") if os.name == "nt" else os.path.join(ENV_DIR, "bin", "pip")

print(f"Environment Python: {ENV_PYTHON}")
print(f"Environment pip:    {ENV_PIP}")
print(f"Exists: {os.path.exists(ENV_PYTHON)}")

Environment Python: C:\Users\salva\.conda\envs\griffin\python.exe
Environment pip:    C:\Users\salva\.conda\envs\griffin\Scripts\pip.exe
Exists: True


---
## 3. Install Dependencies by Layer

We use a mix of **conda install** (for packages with compiled C/Fortran code where conda
provides pre-built binaries) and **pip install** (for packages that are pip-only, like
LangChain). This is standard practice -- conda's solver handles the mix cleanly.

**Rule of thumb:** Install with conda first, then pip for anything conda doesn't have.

### 3a. Scraping & Parsing (conda)

These are the tools we used in Workstream 1 (DHRM scrape) and will reuse for the Workday scrape:

| Package | Why we need it |
|---------|---------------|
| `requests` | HTTP client for the Workday JSON API (POST list endpoint, GET detail endpoint) |
| `beautifulsoup4` | Parse the HTML inside Workday `jobDescription` fields into clean text |
| `pdfplumber` | Extract text from PDF career group pages (6 PDFs in DHRM data) and the FY26 salary PDF |
| `lxml` | Fast XML/HTML parser backend for BeautifulSoup -- significantly faster than the default `html.parser` |

In [4]:
# requests, beautifulsoup4, lxml are all on conda-forge with pre-built binaries.
# pdfplumber is pip-only -- we'll install it with pip below.
subprocess.run(
    [CONDA, "install", "-n", ENV_NAME, "-c", "conda-forge", "-y",
     "requests", "beautifulsoup4", "lxml"],
    check=True,
)
print("\nInstalling pdfplumber via pip (not on conda-forge)...")
subprocess.run(
    [CONDA, "run", "-n", ENV_NAME, "pip", "install", "pdfplumber"],
    check=True,
)


Installing pdfplumber via pip (not on conda-forge)...


CompletedProcess(args=['C:\\ProgramData\\anaconda3\\Scripts\\conda.exe', 'run', '-n', 'griffin', 'pip', 'install', 'pdfplumber'], returncode=0)

### 3b. Data & Database (conda)

| Package | Why we need it |
|---------|---------------|
| `pandas` | Core DataFrame library -- every CSV, every feature matrix, every export goes through pandas |
| `sqlalchemy` | ORM for loading DataFrames into SQLite (WS1) and later MySQL on GCP (Big Data deliverable) |
| `pymysql` | MySQL driver for sqlalchemy -- needed when we migrate from SQLite to Google Cloud SQL |
| `openpyxl` | Read/write `.xlsx` files -- the 7 hr_handoff Excel exports and the task tracker depend on this |

In [6]:
subprocess.run(
    [CONDA, "install", "-n", ENV_NAME, "-c", "conda-forge", "-y",
     "pandas", "sqlalchemy", "pymysql", "openpyxl"],
    check=True,
)

CompletedProcess(args=['C:\\ProgramData\\anaconda3\\Scripts\\conda.exe', 'install', '-n', 'griffin', '-c', 'conda-forge', '-y', 'pandas', 'sqlalchemy', 'pymysql', 'openpyxl'], returncode=0)

### 3c. Visualization (conda)

| Package | Why we need it |
|---------|---------------|
| `streamlit` | The front-end framework for the GRIFFIN web app -- HR pastes a PD, gets classification + pay + reasoning |
| `plotly` | Interactive charts for the Streamlit app (confidence gauges, pay band comparisons) |
| `seaborn` | Statistical plots for the Big Data notebook (roles by family, pay heatmaps, SHAP summaries) |
| `matplotlib` | Base plotting library that seaborn and SHAP both depend on for rendering |

In [6]:
subprocess.run(
    [CONDA, "install", "-n", ENV_NAME, "-c", "conda-forge", "-y",
     "streamlit"],
    check=True,
)

CompletedProcess(args=['C:\\ProgramData\\anaconda3\\Scripts\\conda.exe', 'install', '-n', 'griffin', '-c', 'conda-forge', '-y', 'streamlit'], returncode=0)

### 3d. ML Explainability (conda)

| Package | Why we need it |
|---------|---------------|
| `shap` | SHAP (SHapley Additive exPlanations) values for the H2O model -- required by the Big Data rubric for global + local interpretability. Explains *why* the model predicted a specific career group, which feeds into the LLM's reasoning chain and the fairness guard. |

**Note:** SHAP can be installed now even without H2O. It also works with scikit-learn,
so we can prototype feature engineering with a quick sklearn model before H2O is ready.

In [8]:
subprocess.run(
    [CONDA, "install", "-n", ENV_NAME, "-c", "conda-forge", "-y",
     "shap"],
    check=True,
)

CompletedProcess(args=['C:\\ProgramData\\anaconda3\\Scripts\\conda.exe', 'install', '-n', 'griffin', '-c', 'conda-forge', '-y', 'shap'], returncode=0)

### 3e. LangChain / Agentic AI (pip)

These are **pip-only** packages -- LangChain doesn't publish to conda-forge. This is fine;
conda handles mixed conda+pip environments cleanly as long as we install conda packages
first (which we did above).

| Package | Why we need it |
|---------|---------------|
| `langchain` | High-level agent framework -- provides prompt templates, tool bindings, and agent executors. This is the production framework (Prof. Chung confirmed open-source is acceptable). |
| `langchain-core` | Core abstractions (messages, runnables, output parsers) that `langchain` builds on. Pinned separately so we control the version. |
| `langgraph` | State-machine orchestration for multi-agent workflows. Our three-stage architecture (ML Shortlist -> LLM Classifier -> Confidence Check) is modeled as a LangGraph graph with conditional edges. Satisfies Assignment 5's orchestration component. |

**Design note:** LangChain's `init_chat_model()` gives us LLM-agnostic design for free --
W&M HR can swap between Gemini, Copilot, or Claude by changing a single config string.
No code changes needed. This is a client requirement since W&M uses both Copilot and Gemini.

In [5]:
subprocess.run(
    [CONDA, "run", "-n", ENV_NAME, "pip", "install",
     "langchain", "langchain-core", "langgraph"],
    check=True,
)

CompletedProcess(args=['C:\\ProgramData\\anaconda3\\Scripts\\conda.exe', 'run', '-n', 'griffin', 'pip', 'install', 'langchain', 'langchain-core', 'langgraph'], returncode=0)

---
## 4. Write environment.yml

This file lets anyone on the team recreate the exact same environment with:
```
conda env create -f environment.yml
conda activate griffin
```

It's the conda equivalent of `requirements.txt` -- but better because it captures
both conda and pip packages in a single file, and can specify non-Python deps like Java.

In [ ]:
environment_yml = """# GRIFFIN -- W&M HR Classification Project
# Recreate: conda env create -f environment.yml
# Activate: conda activate griffin

name: griffin
channels:
  - conda-forge
  - defaults

dependencies:
  - python=3.11

  # --- Scraping & Parsing ---
  - requests
  - beautifulsoup4
  - lxml

  # --- Data & Database ---
  - pandas
  - sqlalchemy
  - pymysql
  - openpyxl

  # --- Visualization ---
  - streamlit
  - plotly
  - seaborn
  - matplotlib

  # --- ML Explainability ---
  - shap

  # --- Jupyter kernel support ---
  - ipykernel

  # --- H2O (install separately) ---
  # H2O requires Java. Follow the Big Data course lab setup process.
  # Do NOT uncomment these -- install H2O via the lab instructions instead.

  # --- pip-only packages ---
  - pip:
    - pdfplumber
    - langchain
    - langchain-core
    - langgraph
"""

yml_path = os.path.join(os.getcwd(), "environment.yml")
with open(yml_path, "w") as f:
    f.write(environment_yml)

print(f"Wrote {yml_path}")

---
## 5. Verify All Imports

This cell imports every installed package using the `griffin` environment's Python.
If anything fails, re-run the relevant install cell above.

In [4]:
verify_script = """
import sys
print(f"Python: {sys.executable}")
print(f"Version: {sys.version}")
print()

packages = [
    ("requests",       "requests"),
    ("beautifulsoup4", "bs4"),
    ("pdfplumber",     "pdfplumber"),
    ("lxml",           "lxml"),
    ("pandas",         "pandas"),
    ("sqlalchemy",     "sqlalchemy"),
    ("pymysql",        "pymysql"),
    ("openpyxl",       "openpyxl"),
    ("streamlit",      "streamlit"),
    ("plotly",         "plotly"),
    ("seaborn",        "seaborn"),
    ("matplotlib",     "matplotlib"),
    ("shap",           "shap"),
    ("langchain",      "langchain"),
    ("langchain-core", "langchain_core"),
    ("langgraph",      "langgraph"),
]

passed = 0
failed = 0
for name, module in packages:
    try:
        mod = __import__(module)
        ver = getattr(mod, '__version__', 'OK')
        print(f\"  [PASS] {name:20s} {ver}\")
        passed += 1
    except ImportError as e:
        print(f\"  [FAIL] {name:20s} {e}\")
        failed += 1

print()
print(f\"Results: {passed} passed, {failed} failed out of {len(packages)} packages\")
if failed == 0:
    print(\"All imports OK -- environment is ready.\")
else:
    print(\"Some packages failed -- re-run the install cells above.\")
"""

result = subprocess.run(
    [ENV_PYTHON, "-c", verify_script],
    capture_output=True, text=True,
)
print(result.stdout)
if result.stderr:
    # Filter out conda's own activation messages
    stderr_lines = [l for l in result.stderr.splitlines()
                    if not l.startswith("WARNING") and "conda" not in l.lower()]
    if stderr_lines:
        print("--- stderr ---")
        print("\n".join(stderr_lines))

Python: C:\Users\salva\.conda\envs\griffin\python.exe
Version: 3.11.15 | packaged by Anaconda, Inc. | (main, Mar 11 2026, 17:12:15) [MSC v.1942 64 bit (AMD64)]

  [PASS] requests             2.33.1
  [PASS] beautifulsoup4       4.14.3
  [PASS] pdfplumber           0.11.9
  [PASS] lxml                 6.0.2
  [PASS] pandas               2.3.3
  [PASS] sqlalchemy           2.0.48
  [PASS] pymysql              1.4.6
  [PASS] openpyxl             3.1.5
  [PASS] streamlit            1.55.0
  [PASS] plotly               6.6.0
  [PASS] seaborn              0.13.2
  [PASS] matplotlib           3.10.8
  [PASS] shap                 0.48.0
  [PASS] langchain            1.2.13
  [PASS] langchain-core       1.2.23
  [PASS] langgraph            OK

Results: 16 passed, 0 failed out of 16 packages
All imports OK -- environment is ready.



---
## 6. Register Jupyter Kernel

This makes the `griffin` environment available as a selectable kernel in both places
you run notebooks:

- **VS Code:** Auto-detects conda environments via the Python extension. After this step,
  `griffin` appears in the kernel picker without any extra configuration.
- **Terminal Jupyter (`jupyter notebook`):** Needs a kernel registration entry. The cell
  below installs `ipykernel` into the conda env and registers it as **"GRIFFIN (conda)"**
  in Jupyter's kernel registry at `~\AppData\Roaming\jupyter\kernels\`.

In [ ]:
# Install ipykernel into the griffin env and register it
subprocess.run(
    [CONDA, "install", "-n", ENV_NAME, "-c", "conda-forge", "-y", "ipykernel"],
    check=True,
)
subprocess.run(
    [CONDA, "run", "-n", ENV_NAME, "python", "-m", "ipykernel", "install",
     "--user", "--name", "griffin", "--display-name", "GRIFFIN (conda)"],
    check=True,
)
print("\nKernel registered as 'GRIFFIN (conda)'.")
print("- VS Code: Select 'griffin' from the kernel picker (top-right of any .ipynb).")
print("- Terminal: Run 'jupyter notebook', then pick 'GRIFFIN (conda)' from the New menu.")

---
## 7. What's Next?

With the environment ready, here's the project status and critical path:

| Task | Notebook / File | Status |
|------|----------------|--------|
| **A1: Environment setup** | This notebook | DONE (Mar 30) |
| **A2: Workday scrape** | `griffin_workday_scrape.ipynb` | DONE (Mar 31) -- 103 staff PDs, 47 faculty excluded |
| **A3: Label stripping** | `griffin_workday_scrape.ipynb` | DONE (Mar 31) -- zero label leakage confirmed |
| **A4: Feature engineering** | `griffin_feature_engineering.ipynb` (TBD) | NEXT -- extract structured signals from censored PDs |
| **A5: H2O AutoML training** | TBD | After A4 + H2O setup |
| **B1: LangChain multi-agent** | TBD | After A5 (Assignment 5 deliverable) |

### Key data files produced by A2/A3:
- `workday_training.csv` -- 103 censored PDs with labels (ML training input)
- `workday_postings.csv` -- full dataset with raw text + labels
- `workday_exclusions.csv` -- 47 excluded faculty postings
- `raw_workday/` -- cached JSON (8 list pages + 103 detail pages) -- NEVER re-fetch

### To add H2O:
Follow the **Big Data course lab setup process** for H2O installation into the `griffin` conda environment. The lab walkthrough covers Java dependencies and troubleshooting.

### Quick reference:
```bash
conda activate griffin          # activate the environment
conda deactivate                # deactivate
conda env list                  # see all environments
conda list -n griffin           # see all packages in griffin
conda env export -n griffin > environment.yml  # snapshot exact versions
```